# Importação, Organização e Seleção de Dados com Pandas
**Projeto final:** Endividamento municipal e dependência de transferências da União

IFTO – Campus Palmas       
**CST em Sistemas para Internet**

**Disiciplina:** Fundamentos de Programação Científica (2026/2)
**Aluno:** Marcos Ribeiro Guida

**Hipótese:** municípios mais dependentes do FPM (Fundo de Participação dos Municípios) têm **menor endividamento**, medido pela relação DCL/RCL, do que municípios com maior arrecadação própria.

| Base | Fonte | Formato | O que mede |
|---|---|---|---|
| **A – FPM por município** | Tesouro Transparente | CSV | Dependência: FPM recebido, mês a mês |
| **B – RGF, Anexo 2** | API do SICONFI | JSON → CSV | Endividamento: DC, DCL, RCL e % DCL/RCL |
| **C – DCA, Anexo I-C** | API do SICONFI | JSON → CSV | Receitas: total, FPM, IPTU, ISS |



## 1. Configuração

Os três CSVs ficam em `Meu Drive/Projeto_Endividamento/` e seguem anexos à entrega.

In [ ]:
import os, json, time, threading
import urllib.request, urllib.parse, urllib.error
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

PASTA = '/content/drive/MyDrive/Projeto_Endividamento/'
ARQ_FPM = 'fpm-por-municipio.csv'
ARQ_RGF = 'siconfi_rgf_anexo02_municipios_2025.csv'
ARQ_DCA = 'siconfi_dca_anexoIC_municipios_2025.csv'
TOTAL_MUNICIPIOS = 5570

try:                                    # no Colab, monta o Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:                     # fora do Colab, usa a pasta atual
    PASTA = '../data/raw/'

def caminho(arquivo):
    return PASTA + arquivo

fmt = lambda n: f'{n:,}'.replace(',', '.')

Funções auxiliares usadas nas três bases, a fim de  não repetir código:

In [ ]:
def explorar(df, coluna_municipio=None, marcador_ausente=None):
    # Estrutura da base: tamanho, colunas, tipos, primeiros registros e ausentes.
    print(f'Registros: {fmt(df.shape[0])}  |  Colunas: {df.shape[1]}\n')
    print('Colunas:', list(df.columns), '\n')
    df.info()
    display(df.head())

    nulos = df.isna().sum()
    if nulos.sum() == 0:
        print('Valores ausentes (NaN): nenhum')
    elif nulos.nunique() == 1:
        print(f'Valores ausentes (NaN): {nulos.iloc[0]} em cada uma das {df.shape[1]} colunas '
              f'(linhas totalmente vazias: {df.isna().all(axis=1).sum()})')
    else:
        print('Valores ausentes (NaN) por coluna:', nulos[nulos > 0].to_dict())
    if marcador_ausente:
        texto = df.select_dtypes('object').apply(lambda c: c.astype(str).str.strip().eq(marcador_ausente)).sum()
        print(f'Campos com "{marcador_ausente}" (ausente disfarçado), por coluna:')
        display(texto[texto > 0].to_frame('qtd').T)
    if coluna_municipio:
        n = df[coluna_municipio].nunique()
        print(f'Municípios presentes: {fmt(n)} de {fmt(TOTAL_MUNICIPIOS)} (ausentes: {TOTAL_MUNICIPIOS - n})')

def aplicar_filtros(df, filtros):
    # Aplica os filtros em sequência e mostra quantas linhas cada um remove.
    print(f'{"Linhas iniciais":48s}{fmt(len(df)):>10}')
    for descricao, condicao in filtros:
        antes = len(df)
        df = df[condicao(df)]
        print(f'{descricao:48s}{fmt(len(df)):>10}   (-{fmt(antes - len(df))})')
    return df.reset_index(drop=True)

def relatorio_colunas(original, mantidas):
    # Mostra as colunas mantidas e descartadas.
    descartadas = [c for c in original.columns if c not in mantidas]
    print(f'Colunas originais: {original.shape[1]}  |  mantidas: {len(mantidas)}  |  descartadas: {len(descartadas)}')
    print('Mantidas   :', mantidas)
    print('Descartadas:', descartadas)

## 2. Importação dos dados

### A – FPM (CSV do Tesouro Transparente)
O arquivo usa caractere `;` como separador, codificação `latin1`, e o `COD_MUN` tem zeros à esquerda; por isso esse campo é lido como texto.

In [ ]:
fpm_original = pd.read_csv(caminho(ARQ_FPM), sep=';', encoding='latin1', dtype={'COD_MUN': str})

### B e C – SICONFI (API de Dados Abertos)

A API (`apidatalake.tesouro.gov.br/ords/siconfi/tt/`) devolve JSON. Como ela **exige o código IBGE do município** (`id_ente`), é feita **uma requisição por município**:

1. `/entes` → lista dos 5.570 municípios;
2. `/rgf` → Anexo 2 do RGF de 2025. Municípios com menos de 50 mil habitantes podem entregar o **RGF Simplificado semestral**; a função testa os dois formatos;
3. `/dca` → Anexo I-C da DCA de 2025.

Para não ser bloqueada pela API (erro **HTTP 429**), a extração usa 4 conexões simultâneas, pausa ao receber 429 e guarda cada município em cache. Ela leva de 40 a 60 minutos, então o resultado é salvo em CSV e **só é refeito se o CSV não existir**.

In [ ]:
API = 'https://apidatalake.tesouro.gov.br/ords/siconfi/tt/'
ANO = 2025
_pausa = {'ate': 0.0}
_trava = threading.Lock()

def consultar(endpoint, **params):
    url = API + endpoint + '?' + urllib.parse.urlencode(params)
    for _ in range(12):
        time.sleep(max(0, _pausa['ate'] - time.time()))
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                return json.load(r)['items']
        except urllib.error.HTTPError as e:
            if e.code == 429:                                    # muitas requisições: pausa 30 s
                with _trava:
                    _pausa['ate'] = time.time() + 30
            else:
                time.sleep(5)
        except Exception:
            time.sleep(5)
    raise RuntimeError(f'Falha: {endpoint} {params.get("id_ente")}')

def baixar_rgf(ente):
    quadrimestral = dict(in_periodicidade='Q', nr_periodo=3, co_tipo_demonstrativo='RGF')
    semestral     = dict(in_periodicidade='S', nr_periodo=2, co_tipo_demonstrativo='RGF Simplificado')
    formatos = [semestral, quadrimestral] if (ente['populacao'] or 0) < 50000 else [quadrimestral, semestral]
    for f in formatos:
        itens = consultar('rgf', an_exercicio=ANO, no_anexo='RGF-Anexo 02', co_esfera='M',
                          co_poder='E', id_ente=ente['cod_ibge'], **f)
        if itens:
            return itens
    return []

def baixar_dca(ente):
    return consultar('dca', an_exercicio=ANO, no_anexo='DCA-Anexo I-C', id_ente=ente['cod_ibge'])

def carregar_siconfi(arquivo, baixar):
    # Lê o CSV já extraído; se não existir, extrai da API (com cache) e salva o CSV.
    if os.path.exists(caminho(arquivo)):
        return pd.read_csv(caminho(arquivo), sep=';')
    cache = caminho('cache_' + arquivo.replace('.csv', '') + '/')
    os.makedirs(cache, exist_ok=True)
    entes = [e for e in consultar('entes') if e['esfera'] == 'M']
    def tarefa(ente):
        arq = f'{cache}{ente["cod_ibge"]}.json'
        if not os.path.exists(arq):
            with open(arq, 'w') as f:
                json.dump(baixar(ente), f)
    with ThreadPoolExecutor(4) as ex:
        list(ex.map(tarefa, entes))
    linhas = [item for e in entes for item in json.load(open(f'{cache}{e["cod_ibge"]}.json'))]
    df = pd.DataFrame(linhas)
    df.to_csv(caminho(arquivo), sep=';', index=False)
    return df

rgf_original = carregar_siconfi(ARQ_RGF, baixar_rgf)
dca_original = carregar_siconfi(ARQ_DCA, baixar_dca)

## 3. Estrutura original

### A – FPM

In [ ]:
explorar(fpm_original, coluna_municipio='COD_MUN', marcador_ausente='-')

Registros: 66.848  |  Colunas: 36

Colunas: ['COD_MUN', 'Município', 'UF', 'Município - UF', 'Mês', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026'] 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66848 entries, 0 to 66847
Data columns (total 36 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   COD_MUN         66840 non-null  object 
 1   Município       66840 non-null  object 
 2   UF              66840 non-null  object 
 3   Município - UF  66840 non-null  object 
 4   Mês             66840 non-null  float64
 5   1996            66840 non-null  object 
 6   1997            66840 non-null  object 
 7   1998            66840 non-null  object 
 8   1999            66840 non-null  object 
 9   2000            66840 non-null  object 
 

,COD_MUN,Município,UF,Município - UF,Mês,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
0,0643,Acrelândia,AC,Acrelândia - AC,1.0,"50.880,73","73.719,42","73.220,50","76.781,49","83.690,96","110.764,57","112.856,09","119.175,94","127.983,59","216.910,52","236.023,28","230.306,31","291.567,00","294.030,08","252.598,03","374.002,02","372.184,32","397.659,50","514.429,62","642.730,67","542.154,09","562.990,55","597.039,61","718.583,55","648.961,55","783.665,18","962.058,02","966.919,69","1.201.613,90","1.249.129,95","1.408.936,79"
1,0643,Acrelândia,AC,Acrelândia - AC,2.0,"51.709,80","54.969,17","67.702,95","78.551,03","76.082,65","90.250,27","131.740,35","125.486,80","139.771,00","189.882,04","202.997,98","247.513,45","320.454,53","274.111,92","308.403,99","402.932,34","450.213,47","535.008,34","549.335,60","656.103,47","678.698,50","721.738,74","780.517,49","789.321,00","937.720,47","1.027.186,35","1.342.951,89","1.454.130,96","1.633.895,65","1.676.188,12","1.771.455,60"
2,0643,Acrelândia,AC,Acrelândia - AC,3.0,"44.931,43","50.134,33","57.350,07","80.718,36","82.880,88","78.458,47","100.818,78","108.842,83","113.116,26","182.107,72","191.782,57","201.308,88","252.407,22","219.129,82","229.099,80","263.042,42","305.766,84","308.125,09","325.863,18","477.875,02","412.254,12","452.213,54","525.593,41","590.998,67","548.311,13","688.501,44","814.151,92","889.558,63","1.018.644,35","1.103.847,03","1.072.723,61"
3,0643,Acrelândia,AC,Acrelândia - AC,4.0,"79.159,10","62.869,71","71.586,75","72.990,12","85.965,19","88.703,05","105.297,54","103.632,07","124.600,56","197.717,60","215.914,11","240.689,15","292.572,64","261.577,98","274.495,95","348.661,89","385.366,50","330.611,25","371.917,95","515.753,77","490.081,60","544.840,44","551.067,58","576.584,22","537.466,27","719.776,09","965.058,31","1.017.043,11","1.068.408,96","1.121.696,91","1.263.027,85"
4,0643,Acrelândia,AC,Acrelândia - AC,5.0,"50.923,13","64.160,55","65.075,00","79.041,96","82.224,39","99.733,57","120.683,10","145.356,62","145.116,58","223.168,85","236.961,92","258.050,32","307.834,46","310.741,19","337.960,39","400.190,88","431.005,16","475.043,43","495.626,39","634.233,27","651.690,22","618.027,43","676.293,79","739.945,39","561.530,58","865.071,74","1.073.531,06","1.130.606,69","1.242.957,62","1.430.883,45","1.528.664,67"


Valores ausentes (NaN): 8 em cada uma das 36 colunas (linhas totalmente vazias: 8)


Campos com "-" (ausente disfarçado), por coluna:


,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2026
qtd,7164,768,768,768,768,122,128,130,132,84,84,84,84,72,72,72,72,12,12,12,12,12,12,12,12,12,12,12,12,11140


Municípios presentes: 5.570 de 5.570 (ausentes: 0)


- **Formato largo:** cada linha é um município em um mês, e cada coluna de 1996 a 2026 é o FPM daquele ano (5.570 × 12 = 66.840 linhas).
- Os anos vieram como **texto** (`object`), porque os valores estão no padrão brasileiro (` 50.880,73 `). `Mês` veio como `float64` por causa das linhas vazias.
- **Ausentes:**
  - 8 linhas vazias no fim do arquivo (`NaN`).
  - O hífen `-` significa "sem repasse" e o `isna()` não o detecta:
    - até 2012: municípios ainda não instalados;
    - de 2013 a 2024: Boa Esperança do Norte (MT), instalado em 2025;
    - em 2026: novembro e dezembro, que ainda não ocorreram.

### B – RGF, Anexo 2

In [ ]:
explorar(rgf_original, coluna_municipio='cod_ibge')

Registros: 387.985  |  Colunas: 15

Colunas: ['exercicio', 'periodo', 'periodicidade', 'instituicao', 'cod_ibge', 'uf', 'co_poder', 'populacao', 'anexo', 'esfera', 'rotulo', 'coluna', 'cod_conta', 'conta', 'valor'] 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 387985 entries, 0 to 387984
Data columns (total 15 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   exercicio      387985 non-null  int64  
 1   periodo        387985 non-null  int64  
 2   periodicidade  387985 non-null  object 
 3   instituicao    387985 non-null  object 
 4   cod_ibge       387985 non-null  int64  
 5   uf             387985 non-null  object 
 6   co_poder       387985 non-null  object 
 7   populacao      387985 non-null  int64  
 8   anexo          387985 non-null  object 
 9   esfera         387985 non-null  object 
 10  rotulo         387985 non-null  object 
 11  coluna         387985 non-null  object 
 12  cod_conta      387985 non-null  objec

,exercicio,periodo,periodicidade,instituicao,cod_ibge,uf,co_poder,populacao,anexo,esfera,rotulo,coluna,cod_conta,conta,valor
0,2025,3,Q,Prefeitura Municipal de Arealva - SP,3503406,SP,E,8079,RGF-Anexo 02,M,Padrão,SALDO DO EXERCÍCIO ANTERIOR,DividaConsolidada,DÍVIDA CONSOLIDADA - DC (I),3250711.00
1,2025,3,Q,Prefeitura Municipal de Arealva - SP,3503406,SP,E,8079,RGF-Anexo 02,M,Padrão,Até o 1º Quadrimestre,DividaConsolidada,DÍVIDA CONSOLIDADA - DC (I),3163571.86
2,2025,3,Q,Prefeitura Municipal de Arealva - SP,3503406,SP,E,8079,RGF-Anexo 02,M,Padrão,Até o 2º Quadrimestre,DividaConsolidada,DÍVIDA CONSOLIDADA - DC (I),3087678.11
3,2025,3,Q,Prefeitura Municipal de Arealva - SP,3503406,SP,E,8079,RGF-Anexo 02,M,Padrão,Até o 3º Quadrimestre,DividaConsolidada,DÍVIDA CONSOLIDADA - DC (I),3112052.36
4,2025,3,Q,Prefeitura Municipal de Arealva - SP,3503406,SP,E,8079,RGF-Anexo 02,M,Padrão,SALDO DO EXERCÍCIO ANTERIOR,DividaContratual,Dívida Contratual,3149534.09


Valores ausentes (NaN): nenhum
Municípios presentes: 5.527 de 5.570 (ausentes: 43)


- **Formato longo:** cada linha é um valor do demonstrativo, definido por município (`cod_ibge`), linha do anexo (`cod_conta`) e período (`coluna`).
- Os tipos chegam corretos, porque a origem é JSON.
- **Ausentes:** não há `NaN`. O município que não entregou o RGF de 2025 simplesmente não tem linhas.

In [ ]:
for col in ['exercicio', 'anexo', 'esfera', 'co_poder', 'rotulo', 'periodicidade', 'coluna']:
    print(f'{col:14s}', rgf_original[col].value_counts().head(6).to_dict())

exercicio      {2025: 387985}
anexo          {'RGF-Anexo 02': 387985}
esfera         {'M': 387985}
co_poder       {'E': 387985}
rotulo         {'Padrão': 387985}
periodicidade  {'Q': 256958, 'S': 131027}
coluna         {'SALDO DO EXERCÍCIO ANTERIOR': 106345, 'Até o 3º Quadrimestre': 64743, 'Até o 2º Quadrimestre': 64462, 'Até o 1º Quadrimestre': 64338, 'Até o 2º Semestre': 44081, 'Até o 1º Semestre': 43740}


### C – DCA, Anexo I-C

In [ ]:
explorar(dca_original, coluna_municipio='cod_ibge')

Registros: 712.990  |  Colunas: 11

Colunas: ['exercicio', 'instituicao', 'cod_ibge', 'uf', 'anexo', 'rotulo', 'coluna', 'cod_conta', 'conta', 'valor', 'populacao'] 



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 712990 entries, 0 to 712989
Data columns (total 11 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   exercicio    712990 non-null  int64  
 1   instituicao  712990 non-null  object 
 2   cod_ibge     712990 non-null  int64  
 3   uf           712990 non-null  object 
 4   anexo        712990 non-null  object 
 5   rotulo       712990 non-null  object 
 6   coluna       712990 non-null  object 
 7   cod_conta    712990 non-null  object 
 8   conta        712990 non-null  object 
 9   valor        712990 non-null  float64
 10  populacao    712990 non-null  int64  
dtypes: float64(1), int64(3), object(7)
memory usage: 59.8+ MB


,exercicio,instituicao,cod_ibge,uf,anexo,rotulo,coluna,cod_conta,conta,valor,populacao
0,2025,Prefeitura Municipal de Arealva - SP,3503406,SP,DCA-Anexo I-C,Padrão,Receitas Brutas Realizadas,ReceitasExcetoIntraOrcamentarias,RECEITAS (EXCETO INTRA-ORÇAMENTÁRIAS) (I),65514139.73,8079
1,2025,Prefeitura Municipal de Arealva - SP,3503406,SP,DCA-Anexo I-C,Padrão,Deduções - FUNDEB,ReceitasExcetoIntraOrcamentarias,RECEITAS (EXCETO INTRA-ORÇAMENTÁRIAS) (I),7162607.25,8079
2,2025,Prefeitura Municipal de Arealva - SP,3503406,SP,DCA-Anexo I-C,Padrão,Receitas Brutas Realizadas,RO1.0.0.0.00.0.0,1.0.0.0.00.0.0 - Receitas Correntes,65514139.73,8079
3,2025,Prefeitura Municipal de Arealva - SP,3503406,SP,DCA-Anexo I-C,Padrão,Deduções - FUNDEB,RO1.0.0.0.00.0.0,1.0.0.0.00.0.0 - Receitas Correntes,7162607.25,8079
4,2025,Prefeitura Municipal de Arealva - SP,3503406,SP,DCA-Anexo I-C,Padrão,Receitas Brutas Realizadas,RO1.1.0.0.00.0.0,"1.1.0.0.00.0.0 - Impostos, Taxas e Contribuiçõ...",5352193.43,8079


Valores ausentes (NaN): nenhum
Municípios presentes: 5.506 de 5.570 (ausentes: 64)


In [ ]:
for col in ['exercicio', 'anexo', 'rotulo', 'coluna']:
    print(f'{col:10s}', dca_original[col].value_counts().to_dict())

exercicio  {2025: 712990}
anexo      {'DCA-Anexo I-C': 712990}
rotulo     {'Padrão': 712990}
coluna     {'Receitas Brutas Realizadas': 583428, 'Deduções - FUNDEB': 76690, 'Outras Deduções da Receita': 51991, 'Deduções - Transferências Constitucionais': 881}


Mesmo formato longo do RGF e sem `NaN`. Os municípios ausentes são os que não entregaram a DCA de 2025. Brasília não aparece no RGF nem na DCA, porque o DF declara como ente estadual.

## 4. Seleção dos dados relevantes

Os DataFrames originais (`*_original`) são preservados; as seleções ficam em novos DataFrames (`*_sel`).

### A – FPM

Colunas mantidas: identificação do município, mês e os anos de **2015 a 2025**. Os valores são convertidos de texto para número (`-` vira `NaN`).

In [ ]:
ANOS = [str(a) for a in range(2015, 2026)]
fpm_cols = ['COD_MUN', 'Município', 'UF', 'Mês'] + ANOS
relatorio_colunas(fpm_original, fpm_cols)

fpm_sel = fpm_original[fpm_cols].copy()
fpm_sel[ANOS] = fpm_sel[ANOS].apply(lambda c: pd.to_numeric(
    c.str.strip().str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
    errors='coerce'))

Colunas originais: 36  |  mantidas: 15  |  descartadas: 21
Mantidas   : ['COD_MUN', 'Município', 'UF', 'Mês', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
Descartadas: ['Município - UF', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2026']


In [ ]:
fpm_sel = aplicar_filtros(fpm_sel, [
    ('A1 – remove linhas vazias',               lambda d: d['COD_MUN'].notna()),
    ('A2 – exclui o Distrito Federal',           lambda d: d['UF'] != 'DF'),
    ('A3 – só séries completas de 2015 a 2025',  lambda d: d[ANOS].notna().all(axis=1)
                                                          .groupby(d['COD_MUN']).transform('all')),
])
fpm_sel['Mês'] = fpm_sel['Mês'].astype(int)
fpm_sel.head()

Linhas iniciais                                     66.848
A1 – remove linhas vazias                           66.840   (-8)
A2 – exclui o Distrito Federal                      66.828   (-12)
A3 – só séries completas de 2015 a 2025             66.816   (-12)


,COD_MUN,Município,UF,Mês,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,0643,Acrelândia,AC,1,642730.67,542154.09,562990.55,597039.61,718583.55,648961.55,783665.18,962058.02,966919.69,1201613.90,1249129.95
1,0643,Acrelândia,AC,2,656103.47,678698.50,721738.74,780517.49,789321.00,937720.47,1027186.35,1342951.89,1454130.96,1633895.65,1676188.12
2,0643,Acrelândia,AC,3,477875.02,412254.12,452213.54,525593.41,590998.67,548311.13,688501.44,814151.92,889558.63,1018644.35,1103847.03
3,0643,Acrelândia,AC,4,515753.77,490081.60,544840.44,551067.58,576584.22,537466.27,719776.09,965058.31,1017043.11,1068408.96,1121696.91
4,0643,Acrelândia,AC,5,634233.27,651690.22,618027.43,676293.79,739945.39,561530.58,865071.74,1073531.06,1130606.69,1242957.62,1430883.45


### B – RGF, Anexo 2

Nas bases do SICONFI os filtros vêm **antes** da seleção de colunas, porque usam `coluna`, que depois é descartada.

In [ ]:
FECHAMENTO = ['Até o 3º Quadrimestre', 'Até o 2º Semestre']        # posição em 31/12/2025
CONTAS_RGF = ['DividaConsolidada', 'DividaConsolidadaLiquida', 'RGF2ReceitaCorrenteLiquida',
              'ReceitaCorrenteLiquidaAjustadaParaCalculoDosLimitesDeEndividamento',
              'PercentualDaDCLSobreARCL']
rgf_cols = ['cod_ibge', 'uf', 'populacao', 'periodicidade', 'cod_conta', 'valor']

rgf_sel = aplicar_filtros(rgf_original, [
    ('B1 – só a posição de fechamento do ano',   lambda d: d['coluna'].isin(FECHAMENTO)),
    ('B2 – só as contas do indicador DCL/RCL',  lambda d: d['cod_conta'].isin(CONTAS_RGF)),
])[rgf_cols].copy()
print()
relatorio_colunas(rgf_original, rgf_cols)
rgf_sel.head()

Linhas iniciais                                    387.985
B1 – só a posição de fechamento do ano             108.824   (-279.161)
B2 – só as contas do indicador DCL/RCL              27.141   (-81.683)

Colunas originais: 15  |  mantidas: 6  |  descartadas: 9
Mantidas   : ['cod_ibge', 'uf', 'populacao', 'periodicidade', 'cod_conta', 'valor']
Descartadas: ['exercicio', 'periodo', 'instituicao', 'co_poder', 'anexo', 'esfera', 'rotulo', 'coluna', 'conta']


,cod_ibge,uf,populacao,periodicidade,cod_conta,valor
0,3503406,SP,8079,Q,DividaConsolidada,3112052.36
1,3503406,SP,8079,Q,DividaConsolidadaLiquida,3988074.21
2,3503406,SP,8079,Q,RGF2ReceitaCorrenteLiquida,58351532.48
3,3503406,SP,8079,Q,ReceitaCorrenteLiquidaAjustadaParaCalculoDosLi...,57901532.48
4,3503406,SP,8079,Q,PercentualDaDCLSobreARCL,6.89


### C – DCA, Anexo I-C

In [ ]:
CONTAS_DCA = {
    'TotalReceitas':    'Total das receitas',
    'RO1.0.0.0.00.0.0': 'Receitas correntes',
    'RO1.1.0.0.00.0.0': 'Impostos, taxas e contribuições',
    'RO1.1.1.2.50.0.0': 'IPTU',
    'RO1.1.1.4.51.1.0': 'ISS',
    'RO1.7.0.0.00.0.0': 'Transferências correntes',
    'RO1.7.1.1.51.0.0': 'FPM',
    'RO2.1.0.0.00.0.0': 'Operações de crédito',
}
dca_cols = ['cod_ibge', 'uf', 'populacao', 'cod_conta', 'valor']

dca_sel = aplicar_filtros(dca_original, [
    ('C1 – só receitas brutas (sem deduções)',   lambda d: d['coluna'] == 'Receitas Brutas Realizadas'),
    ('C2 – só as contas dos indicadores',       lambda d: d['cod_conta'].isin(CONTAS_DCA)),
])[dca_cols].copy()
print()
relatorio_colunas(dca_original, dca_cols)
dca_sel.head()

Linhas iniciais                                    712.990


C1 – só receitas brutas (sem deduções)             583.428   (-129.562)
C2 – só as contas dos indicadores                   39.519   (-543.909)

Colunas originais: 11  |  mantidas: 5  |  descartadas: 6
Mantidas   : ['cod_ibge', 'uf', 'populacao', 'cod_conta', 'valor']
Descartadas: ['exercicio', 'instituicao', 'anexo', 'rotulo', 'coluna', 'conta']


,cod_ibge,uf,populacao,cod_conta,valor
0,3503406,SP,8079,RO1.0.0.0.00.0.0,65514139.73
1,3503406,SP,8079,RO1.1.0.0.00.0.0,5352193.43
2,3503406,SP,8079,RO1.1.1.2.50.0.0,1473476.61
3,3503406,SP,8079,RO1.1.1.4.51.1.0,1272305.11
4,3503406,SP,8079,RO1.7.0.0.00.0.0,57029116.23


## 5. Justificativa das escolhas

### 5.1 Qual é o problema que o projeto pretende resolver?

O FPM é a principal receita da maioria dos municípios brasileiros. A LRF e a Resolução nº 40/2001 do Senado limitam a Dívida Consolidada Líquida municipal a 1,2 vez a Receita Corrente Líquida. O projeto investiga se **depender mais do FPM está associado a um endividamento menor**: municípios dependentes teriam menos capacidade de pagamento e menos acesso a crédito, enquanto os de arrecadação própria alta (IPTU, ISS) teriam mais espaço para tomar empréstimos.

Informações a obter, por município:
- **dependência** = FPM ÷ total das receitas (base C), complementada pela série 2015–2025 (base A);
- **arrecadação própria** = impostos, IPTU e ISS (base C);
- **endividamento** = % DCL/RCL em 31/12/2025 (base B).

### 5.2 Por que as colunas selecionadas são importantes?

| Base | Coluna | Contribuição |
|---|---|---|
| A | `COD_MUN` | Identifica o município (código SIAFI); chave para agrupar os meses |
| A | `Município`, `UF` | Leitura dos resultados e comparação entre estados e regiões |
| A | `Mês` | Permite conferir se cada ano está completo e analisar sazonalidade |
| A | `2015`–`2025` | FPM nos anos em que há dados de dívida no SICONFI; 2025 é o último ano completo |
| B, C | `cod_ibge` | Código IBGE: chave de ligação entre RGF, DCA e, depois, o PIB do IBGE |
| B, C | `uf`, `populacao` | Agrupamento regional e controle de porte do município |
| B | `periodicidade` | Indica RGF completo (Q) ou simplificado (S), diferença a controlar na análise |
| B, C | `cod_conta` | Indica qual indicador ou receita é a linha (DCL, RCL, FPM, ISS...) |
| B, C | `valor` | O valor em si: base de todos os indicadores da hipótese |

### 5.3 Por que determinadas colunas foram descartadas?

- **A – `Município - UF`:** é só a junção de duas colunas já mantidas.
- **A – `1996`–`2014`:** não têm DCL/RCL comparável no SICONFI e exigiriam correção monetária.
- **A – `2026`:** ano incompleto (faltam 2 meses); distorceria o FPM anual.
- **B, C – `exercicio`, `anexo`, `esfera`, `co_poder`, `rotulo`:** têm um único valor em toda a base; foram parâmetros da consulta à API.
- **B – `periodo`:** repete `periodicidade`.
- **B, C – `coluna`:** depois dos filtros B1 e C1 fica com um único significado.
- **B, C – `conta`:** descrição longa de `cod_conta`; repete a informação e ocupa muita memória.
- **B, C – `instituicao`:** nome da prefeitura; o `cod_ibge` já identifica o município.

### 5.4 Por que os filtros foram aplicados?

| Filtro | Justificativa |
|---|---|
| **A1** linhas vazias | Rodapé do arquivo; não representam nenhum município |
| **A2** sem DF | Brasília acumula competências de estado e município, e sua dívida é declarada como estadual; não é comparável |
| **A3** séries completas | Todos os municípios precisam ser comparados no mesmo período (remove Boa Esperança do Norte/MT, criado em 2025) |
| **B1** posição de fechamento | O saldo em 31/12 é comparável entre RGF quadrimestral e semestral e com a receita anual da DCA |
| **B2** contas do indicador | DC, DCL, RCL e % DCL/RCL formam a medida de endividamento; o resto é detalhamento |
| **C1** receitas brutas | As deduções (ex.: 20% do FPM para o FUNDEB) mudariam a base de cálculo; bruto com bruto mantém FPM e total comparáveis |
| **C2** contas dos indicadores | Os 8 agregados bastam para dependência, arrecadação própria e operações de crédito |

### 5.5 Quais os benefícios da seleção de dados?

- **Organização:** poucas colunas, todas com papel definido, e as três bases com a mesma lógica (município, conta, valor).
- **Desempenho:** menos memória e processamento nas junções e agregações das próximas etapas (ver seção 6).
- **Qualidade:** sem dupla contagem de períodos ou deduções e sem entes incomparáveis; as comparações ficam justas.
- **Foco:** tudo o que restou responde a uma parte da hipótese.

## 6. Comparação entre os DataFrames originais e selecionados

In [ ]:
mb = lambda df: df.memory_usage(deep=True).sum() / 1024**2

bases = {'A – FPM': (fpm_original, fpm_sel),
         'B – RGF Anexo 2': (rgf_original, rgf_sel),
         'C – DCA Anexo I-C': (dca_original, dca_sel)}

comparacao = pd.DataFrame({
    nome: {'Linhas original': o.shape[0], 'Colunas original': o.shape[1],
           'Linhas selecionado': s.shape[0], 'Colunas selecionado': s.shape[1],
           'Memória original (MB)': round(mb(o), 2), 'Memória selecionado (MB)': round(mb(s), 2),
           'Redução (%)': round((1 - mb(s) / mb(o)) * 100, 1)}
    for nome, (o, s) in bases.items()}).T
comparacao.loc['Total'] = comparacao.sum()
comparacao.loc['Total', 'Redução (%)'] = round((1 - comparacao.loc['Total', 'Memória selecionado (MB)']
                                                / comparacao.loc['Total', 'Memória original (MB)']) * 100, 1)
inteiras = ['Linhas original', 'Colunas original', 'Linhas selecionado', 'Colunas selecionado']
comparacao[inteiras] = comparacao[inteiras].astype(int)
comparacao

,Linhas original,Colunas original,Linhas selecionado,Colunas selecionado,Memória original (MB),Memória selecionado (MB),Redução (%)
A – FPM,66848,36,66816,15,137.36,17.04,87.6
B – RGF Anexo 2,387985,15,27141,6,272.33,5.32,98.0
C – DCA Anexo I-C,712990,11,39519,5,396.89,5.26,98.7
Total,1167823,62,133476,26,806.58,27.62,96.6


In [ ]:
# FPM: o que vem da seleção de colunas e o que vem da conversão de texto para número
print(f'FPM original              : {mb(fpm_original):6.1f} MB')
print(f'FPM só com 15 colunas     : {mb(fpm_original[fpm_cols]):6.1f} MB')
print(f'FPM selecionado (numérico): {mb(fpm_sel):6.1f} MB')

FPM original              :  137.4 MB
FPM só com 15 colunas     :   55.0 MB
FPM selecionado (numérico):   17.0 MB


### Interpretação

- **Resultado geral:** as três bases passam de cerca de 807 MB para 28 MB, uma redução de aproximadamente 97%. Os valores podem variar um pouco conforme a versão do pandas.
- **FPM (≈ 88%):** as linhas praticamente não mudam. Os filtros corrigem *quais* municípios entram, não o volume. A redução vem de duas fontes:
  - **colunas:** de 36 para 15, o que dá cerca de 60% a menos;
  - **tipos:** cada valor em texto é um objeto Python de dezenas de bytes; em `float64` ocupa 8 bytes.
- **RGF e DCA (≈ 98–99%):** em formato longo, o que pesa são as **linhas**. Os filtros retiram os períodos intermediários, as deduções e centenas de contas fora da hipótese. As colunas de texto repetido (`conta`, `instituicao`, `anexo`) completam a redução.
- **Conclusão:** em base larga, ganha-se escolhendo colunas e tipos; em base longa, filtrando linhas. As próximas etapas vão trabalhar com cerca de 28 MB em vez de cerca de 800 MB, o que dá folga dentro do limite de memória do Colab.

## 7. Considerações finais e próximas etapas

As bases selecionadas têm só o que a hipótese precisa:
- **FPM:** 5.568 municípios, de 2015 a 2025, sem ausentes;
- **RGF:** indicadores de dívida em 31/12/2025;
- **DCA:** 8 agregados de receita de 2025.

Próximas etapas:
1. **Pivotar** RGF e DCA (`pivot_table`), com uma linha por município e uma coluna por conta.
2. **Juntar** RGF e DCA pelo `cod_ibge` e calcular:
   - FPM ÷ receita total;
   - arrecadação própria per capita;
   - % DCL/RCL.
3. **Ligar a base de FPM**, que usa código SIAFI e não IBGE, por uma tabela de correspondência SIAFI × IBGE.
4. **Tratar inconsistências:**
   - municípios sem RGF ou sem DCA;
   - municípios sem a linha de Dívida Consolidada;
   - DCL negativa;
   - diferenças entre RGF completo e simplificado.
5. Incluir o **PIB per capita** (IBGE) e fazer estatísticas, agrupamentos por UF e região e visualizações.

In [ ]:
for nome, df in [('fpm_selecionado.csv', fpm_sel), ('rgf_selecionado.csv', rgf_sel), ('dca_selecionado.csv', dca_sel)]:
    df.to_csv(caminho(nome), sep=';', index=False)
print('Bases selecionadas salvas em', PASTA or 'pasta atual')

Bases selecionadas salvas em pasta atual
